In [1]:
import sys
sys.path.insert(0, "..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

from dataset import RecordingDataset
from model import VAE


In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-4
LATENT_DIM = 128
IMG_SIZE = 256

In [3]:
dataset = RecordingDataset(data_dir="../try", game="doom", mode="vae")
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [4]:
model = VAE(
    in_channels=3,
    latent_dim=LATENT_DIM,
    img_size=IMG_SIZE,
    encoder_channels=[32, 64, 128, 256],
    encoder_kernels=[4, 4, 4, 4],
    encoder_strides=[2, 2, 2, 2],
    attention_layers=[2, 3],
    num_attention_heads=4,
    final_activation="sigmoid",
)


In [5]:
optimizer = optim.AdamW(model.parameters(), lr=LR)

In [6]:
from tqdm import tqdm

In [7]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    last_batch = None

    # Создаём прогресс-бар и сохраняем ссылку
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        x = batch.to(DEVICE)

        optimizer.zero_grad()
        recon_x, mu, logvar = model(x)
        loss, _, _ = model.loss_vae(recon_x, x, mu, logvar, beta=1.0)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        last_batch = x

        # Обновляем информацию в строке прогресс-бара
        pbar.set_postfix(loss=loss.item())   # показывает текущий loss для батча

    avg_loss = total_loss / len(dataloader.dataset)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}")

    model.eval()
    with torch.no_grad():
        recon = model(last_batch)[0]
        n = min(4, len(last_batch))
        orig = last_batch[:n].cpu()
        recon_imgs = recon[:n].cpu()

        fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
        for i in range(n):
            axes[0, i].imshow(np.transpose(orig[i].numpy(), (1, 2, 0)))
            axes[0, i].axis("off")
            axes[1, i].imshow(np.transpose(recon_imgs[i].numpy(), (1, 2, 0)))
            axes[1, i].axis("off")
        axes[0, 0].set_ylabel("Оригинал")
        axes[1, 0].set_ylabel("Реконструкция")
        plt.suptitle(f"Epoch {epoch+1}/{EPOCHS}")
        plt.tight_layout()
        plt.savefig(f"../weights/doom/val_epoch_{epoch+1:03d}.png", dpi=150)
        plt.close()

    model.save_pretrained(f"../weights/doom_{epoch}")
    print(f"Модель сохранена в ../weights/doom_{epoch}")
    model.train()


Epoch 1/50:   0%|          | 0/2333 [00:00<?, ?it/s]

Epoch 1/50: 100%|██████████| 2333/2333 [15:53<00:00,  2.45it/s, loss=6.23e+3]


Epoch 1/50, Loss: 3006.2111
Модель сохранена в ../weights/doom_0


Epoch 2/50:  17%|█▋        | 403/2333 [02:45<13:11,  2.44it/s, loss=1.35e+4]  


KeyboardInterrupt: 